# 05 — Nested Functions and Closures

## Functions & Scope

This notebook builds directly on `04_Function_Scope_and_LEGB.ipynb`.

The goal is to understand **why closures work**, not just memorize a definition.

By the end, you will understand nested functions, enclosing scope, returned inner functions, closures, retained variables, and `nonlocal` with closures.


## 1. What Are Nested Functions?

A **nested function** is a function defined inside another function.

```text
outer()
   │
   └── inner()
```


In [ ]:
def outer():
    def inner():
        print("Inside inner function")

    inner()

outer()


## 2. Defining a Function Inside Another Function

An inner function can be defined just like any other function. It is normally accessible only from the scope where it is defined.


In [ ]:
def outer():
    print("Outer function")

    def inner():
        print("Inner function")

    inner()

outer()


In [ ]:
def outer():
    def inner():
        print("Hello")

    inner()

outer()

# inner()  # NameError


## 3. Calling the Inner Function

The inner function can be called from the outer function after it has been defined.

Execution flow:

```text
Call outer()
    ↓
Define inner()
    ↓
Call inner()
    ↓
inner() finishes
    ↓
outer() finishes
```


In [ ]:
def outer():
    def inner():
        print("Hello from inner")

    print("Calling inner...")
    inner()

outer()


## 4. Accessing Enclosing Variables

An inner function can read a variable from its enclosing function.


In [ ]:
def outer():
    message = "Hello from outer"

    def inner():
        print(message)

    inner()

outer()


In [ ]:
def outer(name):
    def inner():
        print(f"Hello, {name}")

    inner()

outer("Alice")


`name` is a parameter of `outer()`, but it is accessible inside `inner()` because `outer()` is the enclosing scope. This is the foundation for closures.


## 5. Nested Functions and Scope

An inner function follows the LEGB rule when looking up names. If a name is not local to the inner function, Python can find it in the enclosing function.


In [ ]:
def outer():
    count = 0

    def inner():
        print(count)

    inner()

outer()


Reading an enclosing variable does not require `nonlocal`. Assignment is different: without `nonlocal`, an assignment creates a local binding in the inner function.


In [ ]:
def outer():
    count = 0

    def inner():
        # count += 1  # Would cause UnboundLocalError
        pass

    inner()

outer()


In [ ]:
def outer():
    count = 0

    def inner():
        nonlocal count
        count += 1
        print(count)

    inner()
    inner()

outer()


## 6. Returning an Inner Function

An outer function can return the **inner function itself**.

`return inner` returns the function object; it does not call it.


In [ ]:
def outer():
    def inner():
        print("Hello from inner")

    return inner

function = outer()
function()


The execution is:

```text
outer()
  ↓
defines inner()
  ↓
returns inner function object
  ↓
outer() finishes
  ↓
function() calls inner()
```


## 7. What Is a Closure?

A **closure** is a function that remembers and can access variables from its enclosing scope even after the enclosing function has finished executing.


In [ ]:
def outer(message):
    def inner():
        print(message)

    return inner

greet = outer("Hello")
greet()


The key sequence is:

```text
outer()
  ↓
creates message
  ↓
creates inner()
  ↓
returns inner
  ↓
outer() finishes
  ↓
greet() still remembers message
```

That retained access to `message` is what makes `inner` a closure.


## 8. How Closures Remember Variables

Closures can retain different values for different returned function instances.


In [ ]:
def multiplier(factor):
    def multiply(number):
        return number * factor

    return multiply


In [ ]:
double = multiplier(2)
triple = multiplier(3)

print(double(10))
print(triple(10))


Output:

```text
20
30
```

Each closure remembers a different value:

```text
double → factor = 2
triple → factor = 3
```


## 9. Closures with Parameters

An enclosing parameter can be captured by a returned inner function.


In [ ]:
def power(exponent):
    def calculate(number):
        return number ** exponent

    return calculate

square = power(2)
cube = power(3)

print(square(5))
print(cube(5))


Output:

```text
25
125
```

Multiple closure instances can retain different enclosing values:

```text
square → exponent = 2
cube   → exponent = 3
```


## 10. Practical Closure Examples

### Counter

A closure can retain changing state between calls.


In [ ]:
def counter():
    count = 0

    def increment():
        nonlocal count
        count += 1
        return count

    return increment

counter1 = counter()

print(counter1())
print(counter1())
print(counter1())


Output:

```text
1
2
3
```

The closure retains access to `count`, while `nonlocal` allows the inner function to modify it.


In [ ]:
counter1 = counter()
counter2 = counter()

print(counter1())
print(counter1())

print(counter2())
print(counter2())


Output:

```text
1
2
1
2
```

Each call to `counter()` creates independent retained state:

```text
counter1 → count: 1 → 2
counter2 → count: 1 → 2
```


### Greeting Factory

A closure can create specialized functions from a shared pattern.


In [ ]:
def greeting(prefix):
    def greet(name):
        return f"{prefix}, {name}!"

    return greet

say_hello = greeting("Hello")
say_welcome = greeting("Welcome")

print(say_hello("Alice"))
print(say_welcome("Bob"))


Each returned function remembers its own `prefix`.


## 11. `nonlocal` with Closures

Without `nonlocal`, an assignment to `count` inside `increment()` makes `count` local to that inner function.


In [ ]:
def counter():
    count = 0

    def increment():
        # count += 1  # UnboundLocalError
        pass

    return increment


The correct version is:

```python
def increment():
    nonlocal count
    count += 1
```

This connects the concepts:

```text
Nested Function
       +
Enclosing Scope
       +
nonlocal
       =
Closure with State
```


In [ ]:
def counter():
    count = 0

    def increment():
        nonlocal count
        count += 1
        return count

    return increment

counter1 = counter()
print(counter1())
print(counter1())


## 12. Closure vs Normal Nested Function

### Normal nested function

```python
def outer():
    def inner():
        print("Hello")

    inner()
```

The inner function is defined and called inside `outer()`.

### Closure

```python
def outer(message):
    def inner():
        print(message)

    return inner
```

Here `inner` accesses an enclosing variable, is returned, and can be called later while retaining access to that variable.

> Not every nested function is necessarily being used as a closure.


## 13. Common Mistakes

### Forgetting to return the inner function


In [ ]:
def outer():
    def inner():
        print("Hello")

    # No return statement

result = outer()
print(result)


Since `outer()` has no `return` statement, it returns `None`. The inner function was defined but not returned for later use.

### Calling instead of returning

These are different:

```python
return inner
```

returns the function object.

```python
return inner()
```

calls the function immediately and returns its result.


In [ ]:
def outer():
    def inner():
        return "Hello"

    return inner

function = outer()
print(function)
print(function())


In [ ]:
def outer():
    def inner():
        return "Hello"

    return inner()

result = outer()
print(result)


For a closure, we normally return the function object:

```python
return inner
```

### Forgetting `nonlocal`

If a closure needs to modify an enclosing variable, use `nonlocal`:

```python
def increment():
    nonlocal count
    count += 1
```


## 14. Summary

### Nested function
A function defined inside another function.

### Enclosing scope
The outer function's scope is an enclosing scope for the inner function.

### Returning an inner function
`return inner` returns the function object so it can be used later.

### Closure
A function that retains access to variables from its enclosing scope even after the enclosing function has finished.

### `nonlocal`
Used when an inner function needs to modify a variable belonging to an enclosing function.

### Factory function
A function that creates and returns specialized functions.

```text
Nested Function
      ↓
Inner function can access
enclosing variables
      ↓
Return inner function
      ↓
Outer function finishes
      ↓
Inner function retains access
      ↓
Closure
```

For retained mutable state:

```text
Enclosing variable
      +
Nested function
      +
nonlocal
      ↓
Closure with State
```

The next notebook is `06_Lambda_and_Higher_Order_Functions.ipynb`.
